# Sentiment Analysis on Threads App Reviews

This notebook analyzes the Threads social media platform dataset to understand user sentiment and trends.

## 1. Data Loading

Load the dataset and inspect initial structure.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import re

# Load dataset
df = pd.read_csv('../data/37000_reviews_of_thread_app.csv')
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

## 2. Data Exploration

Analyze the dataset structure and distributions.

In [ ]:
# Check missing values
print("Missing values:")
print(df.isnull().sum())

# Rating distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='rating')
plt.title('Rating Distribution')
plt.show()

# Date range
df['review_date'] = pd.to_datetime(df['review_date'])
print("Date range:", df['review_date'].min(), "to", df['review_date'].max())

# Sample reviews
print("Sample positive review (rating 5):")
print(df[df['rating'] == 5]['review_description'].iloc[0])
print("\nSample negative review (rating 1):")
print(df[df['rating'] == 1]['review_description'].iloc[0])

## 3. Data Cleaning

Clean the dataset for analysis.

In [ ]:
# Drop unnecessary columns
df_clean = df.drop(columns=['Unnamed: 0', 'source', 'review_id', 'user_name', 'developer_response', 'developer_response_date', 'appVersion', 'laguage_code', 'country_code'])

# Fill missing thumbs_up
df_clean['thumbs_up'] = df_clean['thumbs_up'].fillna(0)

# Combine title and description
df_clean['text'] = df_clean.apply(lambda row: (row['review_title'] + ' ' if pd.notnull(row['review_title']) else '') + row['review_description'], axis=1)
df_clean = df_clean.drop(columns=['review_title', 'review_description'])

# Text cleaning function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text

df_clean['clean_text'] = df_clean['text'].apply(clean_text)

print("Cleaned dataset shape:", df_clean.shape)
df_clean.head()

## 4. Feature Engineering

Prepare features for modeling.

In [ ]:
# Create sentiment labels from ratings
def rating_to_sentiment(rating):
    if rating <= 2:
        return 'negative'
    elif rating == 3:
        return 'neutral'
    else:
        return 'positive'

df_clean['sentiment'] = df_clean['rating'].apply(rating_to_sentiment)

# TF-IDF vectorization
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(df_clean['clean_text'])
y = df_clean['sentiment']

print("Feature matrix shape:", X.shape)
print("Sentiment distribution:")
print(y.value_counts())

## 5. Sentiment Analysis Model Training

Train a classification model.

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Logistic Regression
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

print("Model trained successfully")

## 6. Model Evaluation

Evaluate the trained model.

In [ ]:
# Predictions
y_pred = model.predict(X_test)

# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=['negative', 'neutral', 'positive'])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['negative', 'neutral', 'positive'], yticklabels=['negative', 'neutral', 'positive'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Trend analysis
df_clean['date'] = pd.to_datetime(df_clean['review_date'])
daily_sentiment = df_clean.groupby(df_clean['date'].dt.date)['rating'].mean()
plt.figure(figsize=(12, 6))
daily_sentiment.plot()
plt.title('Daily Average Rating Trend')
plt.xlabel('Date')
plt.ylabel('Average Rating')
plt.show()